# 🌱 Plant Disease Detection Using Deep Learning

This notebook implements a plant disease classification system using Deep Learning and Transfer Learning techniques.

Models developed:

- Custom CNN Model
- EfficientNetB0 Transfer Learning Model

The workflow includes:

- Dataset preparation
- Image preprocessing
- Model training
- Model evaluation
- Saving trained models and class labels

## Import necessary libraries

In [ ]:
import os

# TensorFlow
import tensorflow as tf

# Image Preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Models
from tensorflow.keras.models import Sequential, Model

# Layers
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

# Transfer Learning Models
from tensorflow.keras.applications import (
    MobileNetV2,
    ResNet50,
    EfficientNetB0
)

# Callbacks
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

## 📂 Dataset Preparation

The plant disease dataset is loaded and the required training, validation, and testing directories are configured.

The dataset contains multiple plant disease categories used for image classification.

The dataset paths are verified before starting model training.

In [3]:
TRAIN_DIR = r"New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\train"

VALID_DIR = r"New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\valid"

TEST_DIR = r"test"

print("Train:", os.path.exists(TRAIN_DIR))
print("Validation:", os.path.exists(VALID_DIR))
print("Test:", os.path.exists(TEST_DIR))

Train: True
Validation: True
Test: True


## 🖼️ Image Preprocessing and Data Augmentation

Image preprocessing is performed to prepare the leaf images for deep learning models.

Techniques applied:

- Image resizing to 224 × 224 pixels
- Pixel normalization
- Rotation augmentation
- Zoom augmentation
- Horizontal and vertical flipping
- Brightness adjustment

Data augmentation helps improve model performance and reduce overfitting.

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training Data (with augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2]
)

# Validation Data (only rescaling)
valid_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

validation_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print("Training Images:", train_generator.samples)
print("Validation Images:", validation_generator.samples)
print("Classes:", train_generator.num_classes)

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.


# 🧠 Baseline CNN Model

A custom Convolutional Neural Network (CNN) is developed as a baseline model.

CNN Architecture:

- Conv2D layers for feature extraction
- MaxPooling layers for dimensionality reduction
- Dropout layers to reduce overfitting
- Dense layers for classification
- Softmax output layer for multi-class prediction

The model is trained to classify plant leaf images into 38 disease classes.


In [11]:
# Number of disease classes
num_classes = 38

# CNN Architecture
cnn_model = Sequential([

    # Conv2D Layer 1
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(pool_size=(2,2)),

    # Conv2D Layer 2
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),

    # Conv2D Layer 3
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),

    # Dropout Layer
    Dropout(0.5),

    # Flatten Layer
    Flatten(),

    # Dense Layers
    Dense(128, activation='relu'),
    Dropout(0.5),

    # Softmax Output Layer
    Dense(num_classes, activation='softmax')
])


# Model Summary
cnn_model.summary()


# ==========================================
# Model Compilation
# ==========================================

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# ==========================================
# Model Training
# ==========================================

history = cnn_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)



print("Baseline CNN  Model created successfully!")

c:\Users\harin\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,173,862 (42.62 MB)

 Trainable params: 11,173,862 (42.62 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 2382s 1s/step - accuracy: 0.4184 - loss: 1.9905 - val_accuracy: 0.7214 - val_loss: 0.9626
Epoch 2/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 45988s 25s/step - accuracy: 0.6464 - loss: 1.1312 - val_accuracy: 0.8412 - val_loss: 0.5542
Epoch 3/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 2051s 1s/step - accuracy: 0.7304 - loss: 0.8537 - val_accuracy: 0.8814 - val_loss: 0.3890
Epoch 4/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 3247s 2s/step - accuracy: 0.7818 - loss: 0.6881 - val_accuracy: 0.8919 - val_loss: 0.3388
Epoch 5/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 14580s 8s/step - accuracy: 0.8119 - loss: 0.5893 - val_accuracy: 0.9101 - val_loss: 0.2889
Epoch 6/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 1310s 723ms/step - accuracy: 0.8340 - loss: 0.5125 - val_accuracy: 0.9178 - val_loss: 0.2579
Epoch 7/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 2299s 1s/step - accuracy: 0.8573 - loss: 0.4434 - val_accuracy: 0.9284 - val_loss: 0.2228
Epoch 8/10
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 5009s 3s/step - accuracy: 0.

In [17]:
import pickle

with open("cnn_history.pkl", "wb") as f:
    pickle.dump(history.history, f)

## 💾 Saving Class Labels

The disease class names are extracted from the dataset and saved using pickle.

These class labels are required during prediction to map the model output index to the actual disease name.

In [10]:
import pickle

class_names = list(train_generator.class_indices.keys())

with open("class_names.pkl", "wb") as f:
    pickle.dump(class_names, f)

print("Class names saved")

Class names saved


# ⚡ EfficientNetB0 Transfer Learning Model

EfficientNetB0 is implemented using Transfer Learning.

The pre-trained ImageNet model is used as a feature extractor, and a custom classification layer is added for plant disease prediction.

Architecture:

- EfficientNetB0 Base Model
- Global Average Pooling Layer
- Dense Layer
- Dropout Layer
- Softmax Output Layer

In [7]:
# Number of classes
num_classes = train_generator.num_classes

# Load Pretrained Model
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers
base_model.trainable = False

# Build Model
efficientnet_model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Summary
efficientnet_model.summary()

# Compile
efficientnet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
checkpoint = ModelCheckpoint(
    "efficientnet_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# Train
history_efficientnet = efficientnet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5,
    callbacks=[checkpoint, early_stop]
)

# Save Final Model
efficientnet_model.save("efficientnet_final.keras")

# Evaluate
loss, accuracy = efficientnet_model.evaluate(validation_generator)

print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 38)             │        19,494 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,724,937 (18.02 MB)

 Trainable params: 675,366 (2.58 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

Epoch 1/5
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 0s 700ms/step - accuracy: 0.0280 - loss: 3.6395
Epoch 1: val_accuracy improved from None to 0.02866, saving model to efficientnet_model.keras

Epoch 1: finished saving model to efficientnet_model.keras
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 1576s 864ms/step - accuracy: 0.0280 - loss: 3.6395 - val_accuracy: 0.0287 - val_loss: 3.6361
Epoch 2/5
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 0s 808ms/step - accuracy: 0.0278 - loss: 3.6365
Epoch 2: val_accuracy did not improve from 0.02866
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 1848s 1s/step - accuracy: 0.0278 - loss: 3.6365 - val_accuracy: 0.0287 - val_loss: 3.6361
Epoch 3/5
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 0s 959ms/step - accuracy: 0.0287 - loss: 3.6364
Epoch 3: val_accuracy improved from 0.02866 to 0.02874, saving model to efficientnet_model.keras

Epoch 3: finished saving model to efficientnet_model.keras
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 2096s 1s/step - accuracy: 0.0287 - loss: 3.6364 - val_accuracy: 0.0287 - val_loss: 3.6361
Epoch

In [15]:
print(test_generator.num_classes)
print(len(test_generator.class_indices))
print(test_generator.class_indices["Tomato___healthy"])

38
38
37


In [13]:
import pickle

with open("efficientnet_history.pkl", "wb") as f:
    pickle.dump(history_efficientnet.history, f)

print("EfficientNet history saved")

EfficientNet history saved


## 💾 Saving Final Model Files

The following files are saved for integration with the Streamlit application:

- EfficientNetB0 trained model (`efficientnet_final.keras`)
- Training history (`efficientnet_history.pkl`)
- Disease class names (`class_names.pkl`)

These files are used for real-time plant disease prediction.

In [14]:
# Save EfficientNet model
efficientnet_model.save("efficientnet_final.keras")


# Save EfficientNet training history
import pickle

with open("efficientnet_history.pkl", "wb") as f:
    pickle.dump(history_efficientnet.history, f)


# Save class names (same dataset classes)
class_names = list(train_generator.class_indices.keys())

with open("class_names.pkl", "wb") as f:
    pickle.dump(class_names, f)

print("EfficientNet files saved successfully")

EfficientNet files saved successfully
